In [7]:

from pathlib import Path
import pandas as pd

# Locate the project root from this notebook's working directory.
possible_paths = [
    Path("data/raw/credit_risk_dataset.csv"),
    Path("../data/raw/credit_risk_dataset.csv"),
    Path.cwd() / "data/raw/credit_risk_dataset.csv",
]

DATA_PATH = next(
    (path.resolve() for path in possible_paths if path.exists()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find data/raw/credit_risk_dataset.csv. "
        "Check the project folder and file location."
    )

df = pd.read_csv(DATA_PATH)

print("DATASET INSPECTION")
print("=" * 50)
print(f"File: {DATA_PATH}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nFIRST 5 ROWS")
display(df.head())

print("\nCOLUMN NAMES")
print(df.columns.tolist())

print("\nDATA TYPES")
print(df.dtypes)

print("\nMISSING VALUES")
missing = df.isnull().sum()
missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": (missing / len(df) * 100).round(2)
})
display(missing_summary[missing_summary["missing_count"] > 0])

print("\nDUPLICATE ROWS")
print(df.duplicated().sum())

print("\nTARGET DISTRIBUTION")
if "loan_status" in df.columns:
    display(
        pd.DataFrame({
            "count": df["loan_status"].value_counts(dropna=False),
            "percentage": (
                df["loan_status"].value_counts(normalize=True, dropna=False)
                * 100
            ).round(2)
        })
    )
else:
    print("loan_status column not found. Verify the dataset.")


DATASET INSPECTION
File: C:\Users\Admin\banking-risk-intelligence-platform\data\raw\credit_risk_dataset.csv
Rows: 32,581
Columns: 12
Memory usage: 8.63 MB

FIRST 5 ROWS


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4



COLUMN NAMES
['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length']

DATA TYPES
person_age                      int64
person_income                   int64
person_home_ownership             str
person_emp_length             float64
loan_intent                       str
loan_grade                        str
loan_amnt                       int64
loan_int_rate                 float64
loan_status                     int64
loan_percent_income           float64
cb_person_default_on_file         str
cb_person_cred_hist_length      int64
dtype: object

MISSING VALUES


,missing_count,missing_percent
person_emp_length,895,2.75
loan_int_rate,3116,9.56



DUPLICATE ROWS
165

TARGET DISTRIBUTION


,count,percentage
loan_status,,
0,25473,78.18
1,7108,21.82


Data Dictionary

In [8]:

# STEP 2: Generate the initial data dictionary

data_dictionary = pd.DataFrame({
    "column": df.columns,
    "data_type": [str(df[col].dtype) for col in df.columns],
    "unique_values": [df[col].nunique(dropna=True) for col in df.columns],
    "missing_count": [df[col].isna().sum() for col in df.columns],
    "missing_percent": [
        round(df[col].isna().mean() * 100, 2)
        for col in df.columns
    ]
})

display(data_dictionary)

# Numerical summary
print("NUMERICAL FEATURE SUMMARY")
display(df.describe().T)

# Categorical values
categorical_columns = df.select_dtypes(
    include=["object", "str", "category"]
).columns

print("CATEGORICAL VALUE DISTRIBUTIONS")

for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))

# Potentially suspicious numerical values
print("\nPOTENTIAL DATA QUALITY ISSUES")

checks = {
    "Age below 18": df["person_age"] < 18,
    "Age above 100": df["person_age"] > 100,
    "Negative income": df["person_income"] < 0,
    "Non-positive loan amount": df["loan_amnt"] <= 0,
    "Negative employment length": df["person_emp_length"] < 0,
    "Employment length above age": (
        df["person_emp_length"] > df["person_age"]
    ),
    "Negative credit history length": (
        df["cb_person_cred_hist_length"] < 0
    ),
    "Invalid loan status": ~df["loan_status"].isin([0, 1]),
}

for label, condition in checks.items():
    print(f"{label}: {int(condition.fillna(False).sum())}")

# Review exact duplicate rows without deleting them
print("\nEXACT DUPLICATE ROWS — SAMPLE")
display(df[df.duplicated(keep=False)].head(10))


,column,data_type,unique_values,missing_count,missing_percent
0,person_age,int64,58,0,0.00
1,person_income,int64,4295,0,0.00
2,person_home_ownership,str,4,0,0.00
3,person_emp_length,float64,36,895,2.75
4,loan_intent,str,6,0,0.00
5,loan_grade,str,7,0,0.00
6,loan_amnt,int64,753,0,0.00
7,loan_int_rate,float64,348,3116,9.56
8,loan_status,int64,2,0,0.00
9,loan_percent_income,float64,77,0,0.00


NUMERICAL FEATURE SUMMARY


,count,mean,std,min,25%,50%,75%,max
person_age,32581.0,27.734600,6.348078,20.00,23.00,26.00,30.00,144.00
person_income,32581.0,66074.848470,61983.119168,4000.00,38500.00,55000.00,79200.00,6000000.00
person_emp_length,31686.0,4.789686,4.142630,0.00,2.00,4.00,7.00,123.00
loan_amnt,32581.0,9589.371106,6322.086646,500.00,5000.00,8000.00,12200.00,35000.00
loan_int_rate,29465.0,11.011695,3.240459,5.42,7.90,10.99,13.47,23.22
loan_status,32581.0,0.218164,0.413006,0.00,0.00,0.00,0.00,1.00
loan_percent_income,32581.0,0.170203,0.106782,0.00,0.09,0.15,0.23,0.83
cb_person_cred_hist_length,32581.0,5.804211,4.055001,2.00,3.00,4.00,8.00,30.00


CATEGORICAL VALUE DISTRIBUTIONS

person_home_ownership
person_home_ownership
RENT        16446
MORTGAGE    13444
OWN          2584
OTHER         107
Name: count, dtype: int64

loan_intent
loan_intent
EDUCATION            6453
MEDICAL              6071
VENTURE              5719
PERSONAL             5521
DEBTCONSOLIDATION    5212
HOMEIMPROVEMENT      3605
Name: count, dtype: int64

loan_grade
loan_grade
A    10777
B    10451
C     6458
D     3626
E      964
F      241
G       64
Name: count, dtype: int64

cb_person_default_on_file
cb_person_default_on_file
N    26836
Y     5745
Name: count, dtype: int64

POTENTIAL DATA QUALITY ISSUES
Age below 18: 0
Age above 100: 5
Negative income: 0
Non-positive loan amount: 0
Negative employment length: 0
Employment length above age: 2
Negative credit history length: 0
Invalid loan status: 0

EXACT DUPLICATE ROWS — SAMPLE


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
14,23,115000,RENT,2.0,EDUCATION,A,35000,7.90,0,0.30,N,4
238,22,183000,MORTGAGE,3.0,EDUCATION,A,1000,NaN,0,0.01,N,2
252,24,85000,RENT,5.0,MEDICAL,B,25000,10.62,0,0.29,N,4
591,26,20000,MORTGAGE,5.0,MEDICAL,A,4800,5.99,0,0.24,N,3
740,21,21600,OWN,NaN,VENTURE,A,7125,6.99,0,0.33,N,3
817,22,22000,OWN,1.0,DEBTCONSOLIDATION,D,8500,15.05,0,0.39,N,4
920,26,50000,RENT,10.0,PERSONAL,B,18950,NaN,1,0.38,N,3
1094,22,24449,OWN,2.0,VENTURE,B,1800,11.89,0,0.07,N,4
1142,25,60000,RENT,3.0,PERSONAL,A,17000,6.91,0,0.28,N,2
1453,23,20400,RENT,2.0,MEDICAL,A,1200,7.90,0,0.06,N,2


In [9]:

# Investigate suspicious values without modifying the original data

print("RECORDS WITH AGE ABOVE 100")
display(df[df["person_age"] > 100])

print("\nEMPLOYMENT LENGTH ABOVE AGE")
display(
    df[df["person_emp_length"] > df["person_age"]]
)

print("\nEXTREME EMPLOYMENT LENGTH")
display(
    df[df["person_emp_length"] > 50]
)

print("\nEXTREME INCOME: TOP 10")
display(
    df.nlargest(10, "person_income")[
        ["person_age", "person_income", "loan_amnt",
         "loan_percent_income", "loan_status"]
    ]
)

print("\nMISSING INTEREST RATE BY LOAN STATUS")
display(
    df.groupby("loan_status")["loan_int_rate"]
      .agg(["count", "mean"])
)

print("\nMISSINGNESS BY LOAN STATUS")
display(
    df.assign(
        interest_rate_missing=df["loan_int_rate"].isna()
    ).groupby("loan_status")["interest_rate_missing"]
     .agg(["sum", "mean"])
)


RECORDS WITH AGE ABOVE 100


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
81,144,250000,RENT,4.0,VENTURE,C,4800,13.57,0,0.02,N,3
183,144,200000,MORTGAGE,4.0,EDUCATION,B,6000,11.86,0,0.03,N,2
575,123,80004,RENT,2.0,EDUCATION,B,20400,10.25,0,0.25,N,3
747,123,78000,RENT,7.0,VENTURE,B,20000,NaN,0,0.26,N,4
32297,144,6000000,MORTGAGE,12.0,PERSONAL,C,5000,12.73,0,0.00,N,25



EMPLOYMENT LENGTH ABOVE AGE


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
210,21,192000,MORTGAGE,123.0,VENTURE,A,20000,6.54,0,0.10,N,4



EXTREME EMPLOYMENT LENGTH


...

,person_age,person_income,loan_amnt,loan_percent_income,loan_status
32297,144,6000000,5000,0.00,0
30049,42,2039784,8450,0.00,0
32546,60,1900000,1500,0.00,0
32497,63,1782000,12025,0.01,0
31924,44,1440000,6400,0.00,0
31922,47,1362000,6600,0.00,0
17833,32,1200000,12000,0.01,0
29119,36,1200000,10000,0.01,0
29120,40,1200000,10000,0.01,0
17834,34,948000,2000,0.00,0



MISSING INTEREST RATE BY LOAN STATUS


,count,mean
loan_status,,
0,23001,10.435999
1,6464,13.060207



MISSINGNESS BY LOAN STATUS


,sum,mean
loan_status,,
0,2472,0.097044
1,644,0.090602


In [10]:

# STEP 3: Validate derived features and duplicate patterns
# This cell does not modify df.

import numpy as np
import pandas as pd

audit_df = df.copy()

# 1. Recalculate loan amount as a proportion of income
audit_df["calculated_loan_percent_income"] = (
    audit_df["loan_amnt"] / audit_df["person_income"].replace(0, np.nan)
)

audit_df["ratio_difference"] = (
    audit_df["loan_percent_income"]
    - audit_df["calculated_loan_percent_income"]
).abs()

print("LOAN-TO-INCOME RATIO CHECK")
print("Rows with difference > 0.02:",
      int((audit_df["ratio_difference"] > 0.02).sum()))

display(
    audit_df.nlargest(10, "ratio_difference")[
        ["person_income", "loan_amnt", "loan_percent_income",
         "calculated_loan_percent_income", "ratio_difference"]
    ]
)

# 2. Inspect the exact duplicate count
print("\nEXACT DUPLICATE ROW COUNT:", int(df.duplicated().sum()))

# 3. Compare target distribution among duplicate and non-duplicate rows
audit_df["is_duplicate_after_first"] = df.duplicated()

print("\nTARGET DISTRIBUTION BY DUPLICATE STATUS")
display(
    audit_df.groupby("is_duplicate_after_first")["loan_status"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "default_rate"})
)

# 4. Review target labels among all copies of repeated rows
all_duplicate_copies = df[df.duplicated(keep=False)]

print("\nROWS BELONGING TO REPEATED GROUPS:",
      len(all_duplicate_copies))

print("\nDEFAULT LABEL COUNTS IN REPEATED GROUPS")
display(all_duplicate_copies["loan_status"].value_counts())

# 5. Check for missing interest rates by other variables
print("\nINTEREST-RATE MISSINGNESS BY LOAN GRADE")
display(
    df.assign(rate_missing=df["loan_int_rate"].isna())
      .groupby("loan_grade")["rate_missing"]
      .agg(["sum", "mean"])
      .rename(columns={"sum": "missing_count",
                       "mean": "missing_fraction"})
)


LOAN-TO-INCOME RATIO CHECK
Rows with difference > 0.02: 283


,person_income,loan_amnt,loan_percent_income,calculated_loan_percent_income,ratio_difference
4,54400,35000,0.55,0.643382,0.093382
11873,60350,35000,0.49,0.579950,0.089950
11253,56403,30000,0.45,0.531887,0.081887
13005,63750,35000,0.47,0.549020,0.079020
24150,54400,27050,0.42,0.497243,0.077243
13856,72250,35000,0.41,0.484429,0.074429
40,62050,30000,0.41,0.483481,0.073481
48,66300,30000,0.38,0.452489,0.072489
13182,66300,30000,0.38,0.452489,0.072489
16670,66300,30000,0.38,0.452489,0.072489



EXACT DUPLICATE ROW COUNT: 165

TARGET DISTRIBUTION BY DUPLICATE STATUS


,count,default_rate
is_duplicate_after_first,,
False,32416,0.218688
True,165,0.115152



ROWS BELONGING TO REPEATED GROUPS: 330

DEFAULT LABEL COUNTS IN REPEATED GROUPS


loan_status
0    292
1     38
Name: count, dtype: int64


INTEREST-RATE MISSINGNESS BY LOAN GRADE


,missing_count,missing_fraction
loan_grade,,
A,1003,0.093069
B,1056,0.101043
C,630,0.097553
D,312,0.086045
E,83,0.086100
F,27,0.112033
G,5,0.078125


In [11]:

# STEP 4: Final credit-data consistency checks
# This analysis does not modify the original df.

import numpy as np
import pandas as pd

audit_df = df.copy()

# A. Recalculate loan-to-income ratio
audit_df["calculated_loan_percent_income"] = (
    audit_df["loan_amnt"] / audit_df["person_income"].replace(0, np.nan)
)

audit_df["ratio_difference"] = (
    audit_df["loan_percent_income"]
    - audit_df["calculated_loan_percent_income"]
).abs()

print("RATIO DIFFERENCE SUMMARY")
display(audit_df["ratio_difference"].describe())

print("\nRATIO DISCREPANCIES BY LOAN STATUS")
audit_df["ratio_mismatch"] = audit_df["ratio_difference"] > 0.02

display(
    audit_df.groupby("loan_status")["ratio_mismatch"]
    .agg(["sum", "mean"])
    .rename(columns={"sum": "mismatch_count",
                     "mean": "mismatch_fraction"})
)

# B. Check identical predictor rows with conflicting targets
# Exclude loan_status from the predictor columns.
feature_cols = [c for c in df.columns if c != "loan_status"]

predictor_duplicates = df[
    df.duplicated(subset=feature_cols, keep=False)
].copy()

print("\nROWS WITH IDENTICAL PREDICTORS, INCLUDING REPEATS")
print(len(predictor_duplicates))

if not predictor_duplicates.empty:
    conflicting_groups = (
        predictor_duplicates.groupby(feature_cols, dropna=False)
        ["loan_status"]
        .nunique()
    )

    conflicting_groups = conflicting_groups[conflicting_groups > 1]

    print("Predictor groups with conflicting target labels:",
          len(conflicting_groups))

    print("Rows in conflicting groups:",
          int(
              predictor_duplicates.groupby(feature_cols, dropna=False)
              ["loan_status"]
              .transform("nunique")
              .gt(1)
              .sum()
          ))
else:
    print("No repeated predictor combinations found.")

# C. Inspect the largest ratio discrepancies
print("\nTOP 10 RATIO DISCREPANCIES")
display(
    audit_df.nlargest(10, "ratio_difference")[
        ["person_income", "loan_amnt", "loan_percent_income",
         "calculated_loan_percent_income", "loan_status"]
    ]
)


RATIO DIFFERENCE SUMMARY


count    32581.000000
mean         0.002762
std          0.004094
min          0.000000
25%          0.001111
50%          0.002500
75%          0.003674
max          0.093382
Name: ratio_difference, dtype: float64


RATIO DISCREPANCIES BY LOAN STATUS


,mismatch_count,mismatch_fraction
loan_status,,
0,0,0.000000
1,283,0.039814



ROWS WITH IDENTICAL PREDICTORS, INCLUDING REPEATS
330
Predictor groups with conflicting target labels: 0
Rows in conflicting groups: 0

TOP 10 RATIO DISCREPANCIES


,person_income,loan_amnt,loan_percent_income,calculated_loan_percent_income,loan_status
4,54400,35000,0.55,0.643382,1
11873,60350,35000,0.49,0.579950,1
11253,56403,30000,0.45,0.531887,1
13005,63750,35000,0.47,0.549020,1
24150,54400,27050,0.42,0.497243,1
13856,72250,35000,0.41,0.484429,1
40,62050,30000,0.41,0.483481,1
48,66300,30000,0.38,0.452489,1
13182,66300,30000,0.38,0.452489,1
16670,66300,30000,0.38,0.452489,1


In [12]:
# STEP 5: Final review of suspicious credit records

# 1. Review the mismatch pattern
mismatches = audit_df[audit_df["ratio_mismatch"]].copy()

print("MISMATCHES BY LOAN GRADE")
display(
    mismatches.groupby("loan_grade").size()
    .sort_values(ascending=False)
    .rename("mismatch_count")
    .to_frame()
)

print("\nMISMATCHES BY LOAN INTENT")
display(
    mismatches.groupby("loan_intent").size()
    .sort_values(ascending=False)
    .rename("mismatch_count")
    .to_frame()
)

# 2. Review implausible ages
print("\nRECORDS WITH AGE ABOVE 100")
display(
    df.loc[df["person_age"] > 100]
    [["person_age", "person_income", "person_emp_length",
      "loan_amnt", "loan_status"]]
)

# 3. Review employment length exceeding age
print("\nEMPLOYMENT LENGTH EXCEEDING AGE")
display(
    df.loc[df["person_emp_length"] > df["person_age"]]
    [["person_age", "person_emp_length", "loan_status"]]
)

# 4. Confirm missing values and exact duplicates
print("\nMISSING VALUES")
display(
    df.isna().sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

print("\nEXACT DUPLICATE ROWS AFTER THE FIRST OCCURRENCE")
print(df.duplicated().sum())

MISMATCHES BY LOAN GRADE


,mismatch_count
loan_grade,
D,147
E,49
B,26
C,24
F,23
A,11
G,3



MISMATCHES BY LOAN INTENT


,mismatch_count
loan_intent,
MEDICAL,283



RECORDS WITH AGE ABOVE 100


...


EMPLOYMENT LENGTH EXCEEDING AGE


,person_age,person_emp_length,loan_status
0,22,123.0,1
210,21,123.0,0



MISSING VALUES


,missing_count
loan_int_rate,3116
person_emp_length,895
person_income,0
person_age,0
person_home_ownership,0
loan_intent,0
loan_grade,0
loan_amnt,0
loan_status,0
loan_percent_income,0



EXACT DUPLICATE ROWS AFTER THE FIRST OCCURRENCE
165


In [13]:
# STEP 5: Final review of suspicious credit records

# 1. Review loan-to-income mismatches
mismatches = audit_df[audit_df["ratio_mismatch"]].copy()

print("MISMATCHES BY LOAN GRADE")
display(
    mismatches.groupby("loan_grade")
    .size()
    .sort_values(ascending=False)
    .rename("mismatch_count")
    .to_frame()
)

print("\nMISMATCHES BY LOAN INTENT")
display(
    mismatches.groupby("loan_intent")
    .size()
    .sort_values(ascending=False)
    .rename("mismatch_count")
    .to_frame()
)

# 2. Review implausible ages
print("\nRECORDS WITH AGE ABOVE 100")
display(
    df.loc[df["person_age"] > 100, [
        "person_age", "person_income",
        "person_emp_length", "loan_amnt", "loan_status"
    ]]
)

# 3. Review employment length exceeding age
print("\nEMPLOYMENT LENGTH EXCEEDING AGE")
display(
    df.loc[df["person_emp_length"] > df["person_age"], [
        "person_age", "person_emp_length", "loan_status"
    ]]
)

# 4. Confirm missing values
print("\nMISSING VALUES")
display(
    df.isna().sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

# 5. Confirm exact duplicate count
print("\nEXACT DUPLICATE ROWS AFTER FIRST OCCURRENCE")
print(df.duplicated().sum())

MISMATCHES BY LOAN GRADE


,mismatch_count
loan_grade,
D,147
E,49
B,26
C,24
F,23
A,11
G,3



MISMATCHES BY LOAN INTENT


,mismatch_count
loan_intent,
MEDICAL,283



RECORDS WITH AGE ABOVE 100


...


EMPLOYMENT LENGTH EXCEEDING AGE


,person_age,person_emp_length,loan_status
0,22,123.0,1
210,21,123.0,0



MISSING VALUES


,missing_count
loan_int_rate,3116
person_emp_length,895
person_income,0
person_age,0
person_home_ownership,0
loan_intent,0
loan_grade,0
loan_amnt,0
loan_status,0
loan_percent_income,0



EXACT DUPLICATE ROWS AFTER FIRST OCCURRENCE
165


## Credit Risk Dataset — Validation Findings and Decisions

### 1. Dataset Overview
- Dataset contains 32,581 records and 12 columns.
- Target variable: `loan_status` (0 = non-default, 1 = default).
- Default cases: 7,108 (21.82%).
- Non-default cases: 25,473 (78.18%).

### 2. Missing Values
- `person_emp_length`: 895 missing values (2.75%).
- `loan_int_rate`: 3,116 missing values (9.56%).
- Missing values will be handled during preprocessing. Imputation parameters will be fitted using training data only.

### 3. Duplicate Records
- 165 exact duplicate rows were identified after their first occurrences.
- Repeated predictor combinations were found, with no conflicting target labels.
- Duplicate handling will be finalized before model training and data splitting to avoid evaluation leakage.

### 4. Implausible Values
- Five records contain ages above 100.
- Two records contain employment lengths of 123 years, exceeding the borrowers' ages.
- These records will be flagged for investigation. Implausible values will not be silently corrected or removed from the original raw dataset.

### 5. Loan-to-Income Ratio Consistency
- The recorded `loan_percent_income` was compared with `loan_amnt / person_income`.
- 283 records exceeded the discrepancy threshold of 0.02.
- All identified discrepancies occurred among default cases.
- The original field will be preserved for auditing. A separately calculated ratio will be evaluated during preprocessing, and the discrepancy pattern will be investigated to avoid introducing data leakage or misleading predictive signals.

### 6. Data Integrity Policy
- The original raw CSV will remain unchanged.
- Data cleaning and feature engineering will be performed on a separate working dataset.
- All exclusions, transformations, and imputation decisions will be documented.
- Model evaluation will account for class imbalance and prevent duplicate records from leaking across training and test sets.

### Conclusion
Initial validation identified missing values, duplicate records, implausible ages and employment lengths, and loan-to-income inconsistencies. The dataset is retained for further analysis, subject to documented preprocessing and validation decisions.

In [14]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Missing values:", int(df.isna().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Age > 100:", int((df["person_age"] > 100).sum()))
print(
    "Employment length > age:",
    int((df["person_emp_length"] > df["person_age"]).sum())
)
print("Ratio mismatches:", int(audit_df["ratio_mismatch"].sum()))
print(
    "Conflicting predictor groups:",
    len(conflicting_groups)
)

Rows: 32581
Columns: 12
Missing values: 4011
Exact duplicate rows: 165
Age > 100: 5
Employment length > age: 2
Ratio mismatches: 283
Conflicting predictor groups: 0
